# ML-03 — Observed CTR regression and action ranking

[Open in Colab](https://colab.research.google.com/github/Catacombs666/ML_int-/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

Built on the FlyRank ML Internship dataset — https://flyrank.ai. This notebook publishes aggregate or derived diagnostics only.

**Author: Majhabul Hoque**


In [1]:
import os, sys, subprocess, importlib.util
from pathlib import Path
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/ML_int-')]
ROOT=next((p for p in candidates if (p/'data/raw/content_refresh_anonymized.csv').is_file()),None)
if ROOT is None:
    destination=Path('/content/ML_int-') if Path('/content').exists() else Path.cwd()/'ML_int-'
    subprocess.run(['git','clone','https://github.com/Catacombs666/ML_int-.git',str(destination)],check=True)
    ROOT=destination
if os.environ.get('CAPSTONE_OFFLINE') != '1':
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'work/requirements.txt')],check=True)
sys.path.insert(0,str(ROOT/'work/scripts'))
import json, pandas as pd, numpy as np
import ctr_capstone as study
print('Setup ready: public starter data; no token required; numerical dependencies loaded.')


Setup ready: public starter data; no token required; numerical dependencies loaded.


## Task and label

Regression estimates same-window reference CTR; a derived ranking prioritizes review. This replaces the earlier solar-oriented draft with a FlyRank-aligned question.

In [2]:
d,audit=study.read_data()
print('Label: 100 * clicks_90d / impressions_90d')
print('Eligible observed weighted CTR (%):',round(100*d.clicks_90d.sum()/d.impressions_90d.sum(),4))

Label: 100 * clicks_90d / impressions_90d
Eligible observed weighted CTR (%): 0.31


## Proxy versus outcome

Observed clicks are the measured outcome. The reference gap is a review heuristic; it is not a label proving the action will work.

In [3]:
print('Model target: observed CTR')
print('Action-success label: unavailable')
print('Primary metric: impression-weighted MAE in percentage points')

Model target: observed CTR
Action-success label: unavailable
Primary metric: impression-weighted MAE in percentage points


## Baseline and assumptions

Position-band pooled CTR is the transparent reference. The learned model is chosen on a separate client validation set.

In [4]:
receipt_path=ROOT/'work/outputs/ctr_metrics.json'
if not receipt_path.exists():
    study.run()
r=json.loads(receipt_path.read_text())
print(pd.DataFrame(r['validation']).T[['mae_pp','weighted_mae_pp']].to_string())

                          mae_pp  weighted_mae_pp
position_baseline       0.223734         0.210215
random_forest           0.225627         0.205625
hist_gradient_boosting  0.230750         0.207905
